# Step 2: Filter notebook

In [1]:
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt
import rasterio

import os

from method_a_buffer import extract_buffer_feature

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm

from shapely.geometry import Point, box, LineString

# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)

## Imports

#### Import des segments

In [2]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input/attributs'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

save_filtered_attributes = False

# Load segments GeoDataFrame (with 'segment_id')
print("Loading pedestrian segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_pedestrian_segments.parquet"))
segmented_net = segmented_net.to_crs(operation_crs)

# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):
    if save_filtered_attributes:
        if row['save_format'] == 'parquet':
            if not os.path.exists(f'{output_step2_path}/gpkg_attributs'):
                os.makedirs(f'{output_step2_path}/gpkg_attributs')
            gdf.to_parquet(f"{output_step2_path}/parquet_attributs/{attribute}.parquet")
            print(f"Filtered data saved for attribute: {attribute} in format: {row['save_format']}")
        elif row['save_format'] == 'gpkg':
            if not os.path.exists(f'{output_step2_path}/parquet_attributs'):
                os.makedirs(f'{output_step2_path}/parquet_attributs')
            gdf.to_file(f"{output_step2_path}/gpkg_attributs/{attribute}.gpkg", driver="GPKG")
            print(f"Filtered data saved for attribute: {attribute} in format: {row['save_format']}")
        else:
            if not os.path.exists(f'{output_step2_path}/csv_attributs'):
                os.makedirs(f'{output_step2_path}/csv_attributs')
            gdf.to_csv(f"{output_step2_path}/csv_attributs/{attribute}.csv", index=False)
            print(f"Warning: Unknown save format {row['save_format']} for attribute {attribute}. Data saved as csv.")
    else:
        print("Note : Save option is disabled.")

Loading pedestrian segments...


**Connectivité du réseau**

In [3]:
import geopandas as gpd
import pandas as pd
import numpy as np
import networkx as nx
from shapely.geometry import Point

def compute_connectivity_metrics(
    segmented_net: gpd.GeoDataFrame,
    buffer_m: int = 50,
    compute_betweenness: bool = False,
    betweenness_k: int | None = None,   # échantillonnage pour accélérer (k=100 par ex.)
    crs_meter_epsg: int | None = None,  # si ton GDF est en degrés, projette d'abord (ex. 2056)
) -> gpd.GeoDataFrame:
    """
    Ajoute des métriques de connectivité par segment.

    Paramètres
    ----------
    segmented_net : GeoDataFrame avec colonnes u, v, key, segment_id, geometry, (facultatif: length_m)
    buffer_m : rayon pour les métriques locales
    compute_betweenness : calcule une betweenness approx. (moyenne des betweenness des deux nœuds)
    betweenness_k : échantillonnage de noeuds pour accélérer betweenness (NetworkX), None = exact
    crs_meter_epsg : si fourni et si le CRS n’est pas métrique, reprojette pour les buffers

    Retour
    ------
    GeoDataFrame enrichi avec colonnes:
      - conn_mean_degree, conn_deadend_flag, conn_intersection_flag
      - conn_nodes_in_buffer, conn_edges_in_buffer
      - conn_intersections_in_buffer, conn_deadends_in_buffer
      - conn_branching_in_buffer, conn_beta_local
      - conn_betweenness (si compute_betweenness=True)
    """

    gdf = segmented_net.copy()

    # --- Sécurité CRS métrique pour les buffers
    if crs_meter_epsg is not None and (gdf.crs is None or not gdf.crs.is_projected):
        gdf = gdf.to_crs(crs_meter_epsg)

    # Longueur en mètres si manquante (suppose CRS métrique)
    if "length_m" not in gdf.columns:
        gdf["length_m"] = gdf.geometry.length

    # --- Construire le graphe (MultiGraph pour respecter les multi-arêtes)
    G = nx.MultiGraph()
    # On ajoute les arêtes avec attributs utiles
    for r in gdf.itertuples(index=False):
        G.add_edge(getattr(r, "u"), getattr(r, "v"),
                   key=getattr(r, "key"),
                   segment_id=getattr(r, "segment_id"),
                   length=getattr(r, "length_m"))

    # --- Degrés des nœuds
    node_degree = dict(G.degree())
    nx.set_node_attributes(G, node_degree, "degree")

    # --- Extraire une table des nœuds (u/v) avec géométrie (depuis les extrémités des segments)
    # NB: si tes IDs u/v proviennent d’OSMnx, c’est cohérent ; sinon, on reconstruit via endpoints.
    node_rows = []
    for r in gdf.itertuples(index=False):
        geom = getattr(r, "geometry")
        # start / end
        x0, y0 = geom.coords[0]
        x1, y1 = geom.coords[-1]
        node_rows.append({"node": getattr(r, "u"), "geometry": Point(x0, y0)})
        node_rows.append({"node": getattr(r, "v"), "geometry": Point(x1, y1)})
    nodes_gdf = gpd.GeoDataFrame(node_rows, geometry="geometry", crs=gdf.crs)
    # Conserver une géométrie par node ID (si multiples, on prend la première)
    nodes_gdf = nodes_gdf.drop_duplicates(subset="node", keep="first").reset_index(drop=True)
    # Ajouter le degree
    nodes_gdf["degree"] = nodes_gdf["node"].map(node_degree).fillna(0).astype(int)


    n_sindex = nodes_gdf.sindex

    # Containers résultats par segment
    out = {
        "segment_id": [],
        "conn_mean_degree": [],
        "conn_deadend_flag": [],
        "conn_intersection_flag": [],
        "conn_nodes_in_buffer": [],
        "conn_edges_in_buffer": [],
        "conn_intersections_in_buffer": [],
        "conn_deadends_in_buffer": [],
        "conn_branching_in_buffer": [],
        "conn_beta_local": [],
    }

    # --- (Optionnel) Betweenness des NOEUDS (approx) pour reporter aux segments
    node_bet = None
    if compute_betweenness:
        # Graph simple pondéré par longueur (plus stable que MultiGraph pour centrality)
        H = nx.Graph()
        for u, v, data in G.edges(data=True):
            w = data.get("length", 1.0)
            if H.has_edge(u, v):
                if w < H[u][v]["weight"]:
                    H[u][v]["weight"] = w
            else:
                H.add_edge(u, v, weight=w)

    # Ici k doit être un int (nb de nœuds à échantillonner) ou None
    node_bet = nx.betweenness_centrality(
        H,
        k=betweenness_k,          # <-- ENTIER (ex. 200) ou None pour exact
        weight="weight",
        normalized=True,
        endpoints=False,
        seed=42                    # pour reproductibilité
    )

    # --- spatial index (on peut le garder)
    e_sindex = gdf.sindex
    n_sindex = nodes_gdf.sindex

    # --- Boucle segments (sans colonne _buffer)
    for r in gdf.itertuples(index=False):
        seg_id = getattr(r, "segment_id")
        u, v = getattr(r, "u"), getattr(r, "v")

        # buffer local (évite le problème d'attribut)
        buf = getattr(r, "geometry").buffer(buffer_m)

        minx, miny, maxx, maxy = buf.bounds
        query_geom = box(minx, miny, maxx, maxy)

        deg_u = node_degree.get(u, 0)
        deg_v = node_degree.get(v, 0)
        mean_deg = (deg_u + deg_v) / 2

        deadend_flag = (deg_u == 1) or (deg_v == 1)
        intersection_flag = (deg_u >= 3) or (deg_v >= 3)

        # --- Nœuds dans le buffer
        cand_nodes_idx = list(n_sindex.query(query_geom, predicate='intersects'))
        nodes_in_buf = nodes_gdf.iloc[cand_nodes_idx]
        nodes_in_buf = nodes_in_buf[nodes_in_buf.geometry.intersects(buf)]

        nb_nodes = len(nodes_in_buf)
        nb_intersections = int((nodes_in_buf["degree"] >= 3).sum())
        nb_deadends = int((nodes_in_buf["degree"] == 1).sum())
        branching = int(((nodes_in_buf["degree"] - 2).clip(lower=0)).sum())

        # --- Arêtes dans le buffer (excluant soi-même)
        cand_edges_idx = list(e_sindex.query(query_geom, predicate='intersects'))
        edges_in_buf = gdf.iloc[cand_edges_idx]
        edges_in_buf = edges_in_buf[edges_in_buf.geometry.intersects(buf)]
        nb_edges = int(len(edges_in_buf) - 1)  # retirer le segment courant

        beta_local = nb_edges / max(1, nb_nodes)

        out["segment_id"].append(seg_id)
        out["conn_mean_degree"].append(mean_deg)
        out["conn_deadend_flag"].append(bool(deadend_flag))
        out["conn_intersection_flag"].append(bool(intersection_flag))
        out["conn_nodes_in_buffer"].append(int(nb_nodes))
        out["conn_edges_in_buffer"].append(int(nb_edges))
        out["conn_intersections_in_buffer"].append(int(nb_intersections))
        out["conn_deadends_in_buffer"].append(int(nb_deadends))
        out["conn_branching_in_buffer"].append(int(branching))
        out["conn_beta_local"].append(float(beta_local))

    metrics = pd.DataFrame(out)

    # --- Betweenness reportée au segment (moyenne des deux nœuds) si demandé
    if compute_betweenness and node_bet is not None:
        bet_vals = []
        for r in gdf.itertuples(index=False):
            u, v = getattr(r, "u"), getattr(r, "v")
            b = 0.5 * (node_bet.get(u, 0.0) + node_bet.get(v, 0.0))
            bet_vals.append(b)
        gdf["conn_betweenness"] = bet_vals

    # Fusion finale
    gdf = gdf.merge(metrics, on="segment_id", how="left")

    return gdf


In [4]:
# Ensure geometry column is set to avoid spatial index errors
segmented_net = segmented_net.set_geometry("geometry")
print(segmented_net.columns)

Index(['Largeur', 'Partage_us', 'Objet', 'Revetement', 'Vitesse', 'Zone_mod',
       'Commune', 'Type', 'Pente', 'Classe', 'Nom_voie', 'Franchisse',
       'PP_Feux', 'SHAPE_Leng', 'geometry', 'length', 'segment_id'],
      dtype='object')


In [5]:
def add_uv_columns(gdf):
    gdf = gdf.copy()
    gdf["u"] = gdf.geometry.apply(lambda g: hash(g.coords[0]))   # noeud départ
    gdf["v"] = gdf.geometry.apply(lambda g: hash(g.coords[-1]))  # noeud arrivée
    gdf["key"] = 0  # pas de multi-arêtes, donc clé unique
    return gdf

segmented_net = add_uv_columns(segmented_net)

In [6]:

segmented_net_metrics = compute_connectivity_metrics(
    segmented_net,
    buffer_m=100,
    compute_betweenness=True,
    betweenness_k=200
)

segmented_net_metrics['filtered'] = 1

In [7]:
segmented_net_metrics

,Largeur,Partage_us,Objet,Revetement,Vitesse,Zone_mod,Commune,Type,Pente,Classe,Nom_voie,Franchisse,PP_Feux,SHAPE_Leng,geometry,length,segment_id,u,v,key,length_m,conn_betweenness,conn_mean_degree,conn_deadend_flag,conn_intersection_flag,conn_nodes_in_buffer,conn_edges_in_buffer,conn_intersections_in_buffer,conn_deadends_in_buffer,conn_branching_in_buffer,conn_beta_local,filtered
0,Très étroit,Aucun,Trottoir,Béton bitumineux,50,None,Thônex,Trottoir,2.6,tertiair,Chemin des Tourterelles,Sans,None,18.611510,"LINESTRING (2505952.43 1117556.983, 2505957.52...",18.611510,000000,-6930693752093156798,-5320831420533533266,0,18.611510,0.000000,1.0,True,False,36,34,4,16,4,0.944444,1
1,Pas de trottoir,Aucun,Trottoir,Béton bitumineux,50,None,Thônex,Passage piéton,0.3,tertiair,Voie Marguerite-MARMOUD,Sans,Oui,17.373877,"LINESTRING (2504875.759 1116854.188, 2504891.4...",17.373877,000001,-5650924595398224121,5479560796140918503,0,17.373877,0.000000,2.0,False,False,73,81,19,20,21,1.109589,1
2,Large,Mixité vélos,Trottoir,Pavés,50,None,Genève-Cité,Trottoir,0.3,tertiair,Rue de la Monnaie,Sans,None,57.198671,"LINESTRING (2500026.558 1117819.302, 2500041.9...",50.000000,000002,-4271552768887947945,-8292225703009643650,0,50.000000,0.000000,2.0,False,False,118,147,51,31,57,1.245763,1
3,Large,Mixité vélos,Trottoir,Pavés,50,None,Genève-Cité,Trottoir,0.3,tertiair,Rue de la Monnaie,Sans,None,57.198671,"LINESTRING (2500045.896 1117773.338, 2500047.9...",7.198671,000003,-8292225703009643650,8047137376149879673,0,7.198671,0.000000,1.5,True,False,97,125,47,23,52,1.288660,1
4,Moyen,Aucun,Trottoir,Pavés,0,Zone piétonne,Lancy,Trottoir,1.5,tertiair,Esplanade de Pont-Rouge,Sans,None,45.125485,"LINESTRING (2498574.627 1115881.289, 2498530.0...",45.125485,000004,-6541324296344382923,-5364403646111035711,0,45.125485,0.000000,1.0,True,False,83,102,33,20,42,1.228916,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134065,Pas de trottoir,None,None,None,0,None,Plan-les-Ouates,Voie de service,4.0,secondai,None,None,None,6.136427,"LINESTRING (2498917.956 1113532.063, 2498922.7...",6.136427,134065,3375169246876258826,8769006445882061809,0,6.136427,0.000000,3.0,False,True,38,43,9,9,9,1.131579,1
134066,Pas de trottoir,None,None,None,0,None,Lancy,Voie de service,7.8,secondai,None,None,None,9.836936,"LINESTRING (2497663.661 1116834.636, 2497661.4...",9.836936,134066,-8130421142391585299,-1351949987365497831,0,9.836936,0.000000,3.0,False,True,81,83,19,31,20,1.024691,1
134067,Pas de trottoir,None,Parking,Béton bitumineux,50,None,Carouge,Voie de service,1.1,secondai,None,None,None,9.957378,"LINESTRING (2499292.972 1115548.703, 2499294.5...",9.957378,134067,-7327938675199793618,-401580201136213416,0,9.957378,0.000015,3.0,False,True,96,121,37,23,45,1.260417,1
134068,Pas de trottoir,None,None,None,0,None,Vernier,Voie de service,0.3,secondai,None,None,None,8.728223,"LINESTRING (2496898.393 1119087.889, 2496904.1...",8.728223,134068,-3881790742168696375,2435142573084777007,0,8.728223,0.000000,3.0,False,True,26,33,12,3,13,1.269231,1
